# ygo-alpha-trio-resnet34 (updated -> ConvNeXt-Tiny trio + Label Smoothing)

Notebook này cập nhật theo yêu cầu:
- Đổi backbone trio từ ResNet34 sang **ConvNeXt-Tiny**.
- Dùng **CrossEntropyLoss(label_smoothing=...)** để giảm performance instability.


In [ ]:
# !pip install -q timm albumentations opencv-python-headless scikit-learn tqdm

In [ ]:

import os, random
from dataclasses import dataclass
import cv2, timm, torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm.auto import tqdm


In [ ]:

SEED=42

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


In [ ]:

TRAIN_CSV='data/train.csv'
TRAIN_IMG_DIR='data/train/images'
TEST_IMG_DIR='data/test/images'
SAMPLE_SUB='data/sample_submission.csv'
OUT_DIR='outputs_ygo_alpha_convnext_tiny'
os.makedirs(OUT_DIR, exist_ok=True)

RARITY_TO_ID = {
    'Common':1,'Rare':2,'Super Rare':3,'Ultra Rare':4,
    'Secret Rare':5,'Prismatic Secret Rare':6,
    'Quarter Century Secret Rare':7,'Starlight Rare':8,
}
RARITY_TO_FOIL={1:(0,0,0),2:(1,0,0),3:(0,1,0),4:(1,1,0),5:(1,1,0),6:(1,1,0),7:(1,1,1),8:(1,1,1)}


In [ ]:

CROP_CFG={
  'name':dict(x1=0.06,y1=0.04,x2=0.94,y2=0.13),
  'art': dict(x1=0.10,y1=0.17,x2=0.90,y2=0.56),
  'full':dict(x1=0.00,y1=0.00,x2=1.00,y2=1.00),
}

def crop_by_region(img, region):
    h,w=img.shape[:2]; c=CROP_CFG[region]
    x1,y1=int(c['x1']*w),int(c['y1']*h)
    x2,y2=int(c['x2']*w),int(c['y2']*h)
    return img[y1:y2, x1:x2]


def get_transforms(train=True):
    name_tr=A.Compose([A.Resize(96,384),A.ColorJitter(0.2,0.2,0.25,0.03,p=0.8),A.RandomGamma((80,120),p=0.5),A.Normalize(),ToTensorV2()])
    art_tr=A.Compose([A.Resize(256,256),A.ColorJitter(0.15,0.2,0.2,0.02,p=0.8),A.GaussianBlur((3,5),p=0.2),A.Normalize(),ToTensorV2()])
    full_tr=A.Compose([A.Resize(384,384),A.ColorJitter(0.12,0.15,0.15,0.02,p=0.7),A.RandomGamma((85,115),p=0.4),A.Normalize(),ToTensorV2()])
    name_va=A.Compose([A.Resize(96,384),A.Normalize(),ToTensorV2()])
    art_va=A.Compose([A.Resize(256,256),A.Normalize(),ToTensorV2()])
    full_va=A.Compose([A.Resize(384,384),A.Normalize(),ToTensorV2()])
    return {'name':name_tr,'art':art_tr,'full':full_tr} if train else {'name':name_va,'art':art_va,'full':full_va}


In [ ]:

class TriDataset(Dataset):
    def __init__(self, df, img_dir, tfms, is_test=False):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir; self.tfms=tfms; self.is_test=is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        p=os.path.join(self.img_dir,row['id'])
        img=cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        n=self.tfms['name'](image=crop_by_region(img,'name'))['image']
        a=self.tfms['art'](image=crop_by_region(img,'art'))['image']
        f=self.tfms['full'](image=crop_by_region(img,'full'))['image']
        if self.is_test: return n,a,f,row['id']
        y=int(row['rarity'])-1
        return n,a,f,torch.tensor(y,dtype=torch.long)


In [ ]:

class TrioConvNeXtTiny(nn.Module):
    def __init__(self, pretrained=True, n_classes=8):
        super().__init__()
        model_name='convnext_tiny'
        self.name_encoder=timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.art_encoder=timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.full_encoder=timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        d=self.name_encoder.num_features
        self.head=nn.Sequential(nn.Linear(d*3,1024),nn.GELU(),nn.Dropout(0.25),nn.Linear(1024,n_classes))
    def forward(self,xn,xa,xf):
        fn=self.name_encoder(xn); fa=self.art_encoder(xa); ff=self.full_encoder(xf)
        return self.head(torch.cat([fn,fa,ff],dim=1))


In [ ]:

@dataclass
class CFG:
    n_splits:int=5
    epochs:int=10
    bs:int=8
    lr:float=2e-4
    wd:float=1e-4
    num_workers:int=2
    label_smoothing:float=0.1
cfg=CFG()

def macro_f1(y_true,y_pred):
    return f1_score(y_true,y_pred,average='macro')

def train_one_fold(df, fold, tr_idx, va_idx):
    tr=df.iloc[tr_idx].copy(); va=df.iloc[va_idx].copy()
    tr_dl=DataLoader(TriDataset(tr,TRAIN_IMG_DIR,get_transforms(True),False), batch_size=cfg.bs,shuffle=True,num_workers=cfg.num_workers,pin_memory=True)
    va_dl=DataLoader(TriDataset(va,TRAIN_IMG_DIR,get_transforms(False),False), batch_size=cfg.bs,shuffle=False,num_workers=cfg.num_workers,pin_memory=True)

    model=TrioConvNeXtTiny(pretrained=True,n_classes=8).to(DEVICE)
    criterion=nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.wd)

    best=-1
    best_path=os.path.join(OUT_DIR,f'convnext_tiny_trio_fold{fold}.pth')
    for ep in range(cfg.epochs):
        model.train()
        for n,a,f,y in tr_dl:
            n,a,f,y=n.to(DEVICE),a.to(DEVICE),f.to(DEVICE),y.to(DEVICE)
            optimizer.zero_grad(); logits=model(n,a,f); loss=criterion(logits,y); loss.backward(); optimizer.step()
        model.eval(); yp=[]; yt=[]
        with torch.no_grad():
            for n,a,f,y in va_dl:
                logits=model(n.to(DEVICE),a.to(DEVICE),f.to(DEVICE))
                yp.extend(torch.argmax(logits,1).cpu().numpy().tolist()); yt.extend(y.numpy().tolist())
        f1=macro_f1(yt,yp)
        print(f'[fold {fold}] epoch {ep}: f1={f1:.4f}')
        if f1>best:
            best=f1; torch.save(model.state_dict(),best_path)
    return best_path


In [ ]:

train_df=pd.read_csv(TRAIN_CSV)
if train_df['rarity'].dtype==object:
    train_df['rarity']=train_df['rarity'].map(RARITY_TO_ID)

skf=StratifiedKFold(n_splits=cfg.n_splits,shuffle=True,random_state=SEED)
model_paths=[]
for fold,(tr_idx,va_idx) in enumerate(skf.split(train_df,train_df['rarity'])):
    model_paths.append(train_one_fold(train_df, fold, tr_idx, va_idx))
print(model_paths)


In [ ]:

def load_models(paths):
    ms=[]
    for p in paths:
        m=TrioConvNeXtTiny(pretrained=False,n_classes=8).to(DEVICE)
        m.load_state_dict(torch.load(p,map_location=DEVICE)); m.eval(); ms.append(m)
    return ms

def predict_rarity(test_df, paths):
    dl=DataLoader(TriDataset(test_df,TEST_IMG_DIR,get_transforms(False),True), batch_size=cfg.bs,shuffle=False,num_workers=cfg.num_workers,pin_memory=True)
    ms=load_models(paths)
    out_ids=[]; out_r=[]
    with torch.no_grad():
        for n,a,f,ids in tqdm(dl,desc='predict'):
            n,a,f=n.to(DEVICE),a.to(DEVICE),f.to(DEVICE)
            probs=[torch.softmax(m(n,a,f),dim=1).cpu().numpy() for m in ms]
            p=np.mean(probs,axis=0)
            pred=np.argmax(p,axis=1)+1
            out_ids.extend(ids); out_r.extend(pred.tolist())
    return pd.DataFrame({'id':out_ids,'rarity':out_r})

sample=pd.read_csv(SAMPLE_SUB)
test_df=sample[['id']].copy()
pred=predict_rarity(test_df, model_paths)
pred['name_foil']=pred['rarity'].map(lambda r:RARITY_TO_FOIL[int(r)][0])
pred['art_foil']=pred['rarity'].map(lambda r:RARITY_TO_FOIL[int(r)][1])
pred['full_foil']=pred['rarity'].map(lambda r:RARITY_TO_FOIL[int(r)][2])
sub=pred[['id','rarity','name_foil','art_foil','full_foil']]
sub.to_csv(os.path.join(OUT_DIR,'submission.csv'),index=False)
sub.head()
